# Lesson 8: n-step A2C on CartPole

This notebook introduces n-step bootstrapping.

Earlier A2C updated after a full episode. n-step A2C updates after a short rollout of `N_STEPS`.

That means the target is:

```text
n-step return = rewards over the next n steps + bootstrapped critic value
```

This is closer to how many scalable RL systems collect rollouts.


## 1) Imports


In [ ]:
%pip install -U "gymnasium[classic-control]"

import random
from collections import deque

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

print("gymnasium:", gym.__version__)
print("torch:", torch.__version__)


## 2) Seeds and Device


In [ ]:
SEED = 1234

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 3) Environment


In [ ]:
train_env = gym.make("CartPole-v1")
test_env = gym.make("CartPole-v1")

train_env.action_space.seed(SEED)
test_env.action_space.seed(SEED + 1)

state, info = train_env.reset(seed=SEED)
print("example state:", state)
print("observation space:", train_env.observation_space)
print("action space:", train_env.action_space)


## 4) Actor-Critic Network


In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
        )
        self.actor = nn.Linear(hidden_dim, output_dim)
        self.critic = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        features = self.shared(x)
        logits = self.actor(features)
        value = self.critic(features).squeeze(-1)
        return logits, value


## 5) Build Policy and Optimizer


In [ ]:
INPUT_DIM = train_env.observation_space.shape[0]
HIDDEN_DIM = 128
OUTPUT_DIM = train_env.action_space.n

policy = ActorCritic(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM).to(device)
optimizer = optim.Adam(policy.parameters(), lr=5e-4)

print(policy)


## 6) n-step Return

If the rollout ends because the episode ended, there is no future value to bootstrap from.

If the rollout stops after `N_STEPS` while the episode is still alive, we bootstrap from the critic:

```text
R = V(s_after_rollout)
```

Then we walk backward through the collected rewards.


In [ ]:
def compute_n_step_returns(rewards, values, bootstrap_value, discount_factor):
    returns = []
    running_return = bootstrap_value

    for reward in reversed(rewards):
        running_return = reward + discount_factor * running_return
        returns.insert(0, running_return)

    returns = torch.stack(returns)
    values = torch.stack(values)
    advantages = returns - values.detach()

    if len(advantages) > 1:
        std = advantages.std(unbiased=False)
        if std > 1e-8:
            advantages = (advantages - advantages.mean()) / (std + 1e-8)

    return returns, advantages, values


## 7) Collect One n-step Rollout

This collects at most `N_STEPS`, not necessarily a full episode.


In [ ]:
def collect_rollout(env, policy, state, n_steps):
    policy.train()

    log_probs = []
    values = []
    rewards = []
    entropies = []
    total_reward = 0.0
    terminated = False
    truncated = False

    for _ in range(n_steps):
        state_tensor = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        logits, value = policy(state_tensor)
        distribution = Categorical(logits=logits)
        action = distribution.sample()
        log_prob = distribution.log_prob(action).squeeze(0)
        entropy = distribution.entropy().squeeze(0)

        next_state, reward, terminated, truncated, info = env.step(action.item())

        log_probs.append(log_prob)
        values.append(value.squeeze(0))
        rewards.append(torch.as_tensor(reward, dtype=torch.float32, device=device))
        entropies.append(entropy)
        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    done = terminated or truncated
    return state, done, log_probs, values, rewards, entropies, total_reward


## 8) Update From n-step Rollout


In [ ]:
def update_from_rollout(log_probs, values, rewards, entropies, bootstrap_value, discount_factor, optimizer):
    returns, advantages, values = compute_n_step_returns(
        rewards, values, bootstrap_value, discount_factor
    )

    log_probs = torch.stack(log_probs)
    entropies = torch.stack(entropies)

    policy_loss = -(advantages.detach() * log_probs).mean()
    value_loss = F.mse_loss(values, returns.detach())
    entropy_bonus = entropies.mean()
    loss = policy_loss + 0.5 * value_loss - 0.01 * entropy_bonus

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=0.5)
    optimizer.step()

    return policy_loss.item(), value_loss.item()


## 9) Train One Episode With n-step Updates


In [ ]:
def train_one_episode(env, policy, optimizer, discount_factor, n_steps, seed=None):
    state, info = env.reset(seed=seed)
    episode_done = False
    episode_reward = 0.0
    policy_losses = []
    value_losses = []

    while not episode_done:
        state, episode_done, log_probs, values, rewards, entropies, rollout_reward = collect_rollout(
            env, policy, state, n_steps
        )
        episode_reward += rollout_reward

        if episode_done:
            bootstrap_value = torch.zeros((), dtype=torch.float32, device=device)
        else:
            with torch.no_grad():
                state_tensor = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
                _, bootstrap_value = policy(state_tensor)
                bootstrap_value = bootstrap_value.squeeze(0)

        policy_loss, value_loss = update_from_rollout(
            log_probs, values, rewards, entropies, bootstrap_value, discount_factor, optimizer
        )
        policy_losses.append(policy_loss)
        value_losses.append(value_loss)

    return np.mean(policy_losses), np.mean(value_losses), episode_reward


## 10) Evaluate


In [ ]:
def evaluate(env, policy, seed=None):
    policy.eval()

    episode_reward = 0.0
    state, info = env.reset(seed=seed)
    terminated = False
    truncated = False

    while not (terminated or truncated):
        state_tensor = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

        with torch.no_grad():
            logits, value = policy(state_tensor)
            action = torch.argmax(logits, dim=-1).item()

        state, reward, terminated, truncated, info = env.step(action)
        episode_reward += reward

    return episode_reward


## 11) Training Loop


In [ ]:
MAX_EPISODES = 500
DISCOUNT_FACTOR = 0.99
N_STEPS = 5
N_TRIALS = 25
REWARD_THRESHOLD = 475
PRINT_EVERY = 10

train_rewards = []
test_rewards = []
recent_test_rewards = deque(maxlen=N_TRIALS)

for episode in range(1, MAX_EPISODES + 1):
    policy_loss, value_loss, train_reward = train_one_episode(
        train_env, policy, optimizer, DISCOUNT_FACTOR, N_STEPS, seed=SEED + episode
    )
    test_reward = evaluate(test_env, policy, seed=SEED + 10_000 + episode)

    train_rewards.append(train_reward)
    test_rewards.append(test_reward)
    recent_test_rewards.append(test_reward)

    if episode % PRINT_EVERY == 0:
        print(
            f"| Episode: {episode:3} | "
            f"Mean Train: {np.mean(train_rewards[-N_TRIALS:]):6.1f} | "
            f"Mean Test: {np.mean(recent_test_rewards):6.1f} | "
            f"Policy Loss: {policy_loss:8.3f} | Value Loss: {value_loss:8.3f} |"
        )

    if len(recent_test_rewards) == N_TRIALS and np.mean(recent_test_rewards) >= REWARD_THRESHOLD:
        print(f"Reached reward threshold in {episode} episodes")
        break


## 12) Plot Rewards


In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(train_rewards, label="Train Reward", alpha=0.7)
plt.plot(test_rewards, label="Test Reward")
plt.axhline(REWARD_THRESHOLD, color="red", linestyle="--", label="Threshold")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.legend()
plt.grid(True)
plt.show()


## 13) Watch Policy


In [ ]:
def watch_policy(policy, seed=SEED, max_steps=500):
    render_env = gym.make("CartPole-v1", render_mode="human")
    state, info = render_env.reset(seed=seed)
    terminated = False
    truncated = False
    total_reward = 0.0
    steps = 0

    while not (terminated or truncated) and steps < max_steps:
        state_tensor = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            logits, value = policy(state_tensor)
            action = torch.argmax(logits, dim=-1).item()
        state, reward, terminated, truncated, info = render_env.step(action)
        total_reward += reward
        steps += 1

    print(f"Episode finished. Total reward: {total_reward}, steps: {steps}")
    render_env.close()


# Uncomment after training if your machine supports GUI rendering.
# watch_policy(policy)


## 14) Exercises

1. Try `N_STEPS = 1`, `5`, and `20`. What changes?
2. Explain when `bootstrap_value` is zero.
3. Why might n-step updates be more scalable than waiting for full episodes?

## 15) Mental Model

n-step A2C sits between one-step TD and full-episode Monte Carlo.

```text
use a few real rewards, then ask the critic to estimate the rest
```
